# Forget-MI LoKU — Machine Unlearning Pipeline

Notebook thực hiện toàn bộ pipeline:

1. **Cell 1** — Mount Drive + pull code mới (KHÔNG xóa data đã extract)
2. **Cell 2** — Extract data & models (chỉ chạy LẦN ĐẦU)
3. **Cell 3** — Preprocess (chỉ chạy LẦN ĐẦU)
4. **Cell 3.5** — Verify config (mỗi lần chạy exp mới)
5. **Cell 4** — Huấn luyện LoKU Unlearning (với **auto-tracking**)
6. **Cell 5** — **Auto-commit & push** kết quả lên GitHub

## Workflow cho mỗi experiment mới

### Lần đầu chạy (full setup)

1. Sửa `config.yaml` ở local → push lên GitHub
2. Mở Colab → chạy **Cell 1 → Cell 2 → Cell 3** (lần đầu, đợi ~3 phút)
3. **Sửa `EXP_NAME` + `HYPOTHESIS`** ở đầu Cell 4
4. Chạy **Cell 3.5 → Cell 4 → Cell 5**

### Lần thứ 2 trở đi (skip extract)

1. Sửa `config.yaml` ở local → push lên GitHub
2. Colab: **Cell 1** (pull code mới, ~10s)
3. **BỎ QUA Cell 2 + Cell 3** (data đã có)
4. Sửa `EXP_NAME` + `HYPOTHESIS` ở Cell 4
5. **Cell 3.5 → Cell 4 → Cell 5**

### Sau khi Colab xong

6. Local: `git pull` để lấy file MD về
7. Điền 3 section vào file MD: Observations / Conclusion / Next steps
8. `git push`

> Lần đầu setup Cell 5: tạo file `/content/drive/MyDrive/Forget-MI-Project/.git-secrets.json` (xem hướng dẫn trong cell).

In [14]:
# ====================================
# CELL 1: Kết nối Drive & Pull Code (giữ data, không clone lại)
# ====================================
from google.colab import drive
import os

# 1. Mount Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("✅ Google Drive đã được kết nối!")

# 2. Pull code mới (KHÔNG xóa thư mục → data đã extract được giữ nguyên)
%cd /content
REPO = "Forget-MI-LoKU"
REPO_URL = "https://github.com/nhnhu146/Forget-MI-LoKU.git"

if not os.path.exists(REPO):
    print(f"🔽 Clone lần đầu: {REPO}")
    !git clone {REPO_URL}
else:
    print(f"🔄 Pull code mới (giữ data đã extract)")
    %cd {REPO}
    !git fetch origin
    !git reset --hard origin/master 2>&1 | tail -3
    %cd /content

%cd {REPO}
!git log --oneline -1

# 3. Cài đặt thư viện (chỉ chạy lần đầu hoặc khi cần update)
import importlib.util
need_install = importlib.util.find_spec("peft") is None or importlib.util.find_spec("pydicom") is None
if need_install:
    print("📦 Cài đặt thư viện...")
    !pip install -q pydicom scikit-image wandb pyyaml pandas
    !pip install -q "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"
else:
    print("✅ Thư viện đã cài, bỏ qua.")

print("\n✅ Môi trường và mã nguồn đã sẵn sàng!")
print("ℹ️  Lần đầu: chạy Cell 2 → Cell 3 (extract data).")
print("ℹ️  Lần sau: bỏ qua Cell 2 + Cell 3, đi thẳng Cell 3.5 → Cell 4 → Cell 5.")

✅ Google Drive đã được kết nối!
/content
🔄 Pull code mới (giữ data đã extract)
/content/Forget-MI-LoKU
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.52 KiB | 1.52 MiB/s, done.
From https://github.com/nhnhu146/Forget-MI-LoKU
   093a3fd..a907fc8  master     -> origin/master
HEAD is now at a907fc8 Cell 1: pull thay vì clone để giữ data; skip cài lại thư viện
/content
/content/Forget-MI-LoKU
a907fc8 (HEAD -> master, origin/master, origin/HEAD) Cell 1: pull thay vì clone để giữ data; skip cài lại thư viện
✅ Thư viện đã cài, bỏ qua.

✅ Môi trường và mã nguồn đã sẵn sàng!
ℹ️  Lần đầu: chạy Cell 2 → Cell 3 (extract data).
ℹ️  Lần sau: bỏ qua Cell 2 + Cell 3, đi thẳng Cell 3.5 → Cell 4 → Cell 5.


In [15]:
# ====================================
# CELL 2: Giải nén Data & Models
# ====================================
!python setup_data.py

🔍 Đang quét Drive để tìm file zip...
✅ Đã tìm thấy thư mục dự án tại: /content/drive/MyDrive/Forget-MI-Project

--- Bắt đầu giải nén ---
📦 Đang giải nén data.zip -> ./data
  🚚 Phát hiện lồng: data/data/ -> Đang gỡ...
  ✅ Gỡ lồng data/ thành công!
📦 Đang giải nén base_model.zip -> ./forgetme/training_original_model
  🚚 Phát hiện lồng: training_original_model/training_original_model/ -> Đang gỡ...
  ✅ Gỡ lồng training_original_model/ thành công!
📦 Đang giải nén retrained_model.zip -> ./model_retrained_3per
  🚚 Phát hiện lồng: model_retrained_3per/model_retrained_3per/ -> Đang gỡ...
  ✅ Gỡ lồng model_retrained_3per/ thành công!

--- Kiểm tra kết quả ---
✅ Text data: ./data/text_data
✅ Image data: ./data/img_data
✅ Base Model: ./forgetme/training_original_model/pytorch_model.bin
✅ Retrained Model: ./model_retrained_3per/pytorch_model.bin

🚀 Tất cả Dữ liệu & Model đã sẵn sàng!


In [16]:
# ====================================
# CELL 3: Tiền xử lý & Thiết lập Output
# ====================================
import os
import shutil

# Tạo all_data.tsv từ các file báo cáo
!python make_tsv.py

# Xóa cache features cũ (nếu có)
!rm -f ./data/metadata/cachedfeatures_train_seqlen-*
!rm -f ./data/metadata/cachednoisyfeatures_train_seqlen-*

# Kết nối thư mục Output với Drive để lưu bền vững
DRIVE_RESULTS = "/content/drive/MyDrive/Forget-MI-Project/unlearning_output"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if os.path.exists("unlearning_output"):
    if os.path.islink("unlearning_output"): os.unlink("unlearning_output")
    else: shutil.rmtree("unlearning_output")

!ln -s "{DRIVE_RESULTS}" ./unlearning_output
print(f"\n✅ Output sẽ được lưu tại: {DRIVE_RESULTS}")

📂 Sử dụng text_data tại: ./data/text_data
Đang đọc file split để lấy mapping study_id -> severity...
Đã nạp 12168 mapping study_id vào bộ nhớ.
Đang gộp file văn bản thành all_data.tsv...
  Đã quét 20000 file... (Khớp 548 báo cáo)
  Đã quét 40000 file... (Khớp 1242 báo cáo)
  Đã quét 60000 file... (Khớp 1720 báo cáo)
  Đã quét 80000 file... (Khớp 2255 báo cáo)
  Đã quét 100000 file... (Khớp 2848 báo cáo)
  Đã quét 120000 file... (Khớp 3370 báo cáo)
  Đã quét 140000 file... (Khớp 3854 báo cáo)
  Đã quét 160000 file... (Khớp 4323 báo cáo)
  Đã quét 180000 file... (Khớp 4836 báo cáo)
  Đã quét 200000 file... (Khớp 5346 báo cáo)
  Đã quét 220000 file... (Khớp 5920 báo cáo)
✅ HOÀN THÀNH! Đã trích xuất 6084 báo cáo trong 4.82 giây.
   File TSV: ./data/metadata/all_data.tsv

✅ Output sẽ được lưu tại: /content/drive/MyDrive/Forget-MI-Project/unlearning_output


In [17]:
# ====================================
# CELL 3.5: Verify config — confirm code mới nhất từ GitHub
# ====================================
# Chạy cell này TRƯỚC Cell 4 để chắc chắn config đúng với exp đang định chạy.
# Nếu thấy giá trị CŨ → bạn quên push từ local, hãy push rồi rerun Cell 1.

print("📋 Config hiện tại (các tham số hay đổi giữa các exp):\n")
!grep -E "^\s*(forget_margin|eta_re_anchor|alpha|beta|theta|gamma|lora_r|lora_alpha|lora_target_modules|use_noise|unlearn_epochs|learning_rate|kappa_cls_retain|kappa_cls_forget|cls_forget_clamp|unfreeze_classifier_heads):" -A 1 config.yaml | grep -v "^--"

print("\n🧪 Kiểm tra code có fix Exp 03 (unfreeze classifier + NegGrad loss):")
!grep -c "Unfroze classifier\|L_cls_ret\|L_cls_frg" training/forgetmi_loku.py | xargs -I{} echo "   → Số lần khớp pattern fix: {} (phải > 5 nếu fix đã merge)"

print("\n🔍 Git commit đang chạy:")
!git log --oneline -1

print("\n👉 Nếu config KHÔNG đúng với exp bạn định chạy:")
print("   1. Local: kiểm tra `git status` xem đã commit chưa")
print("   2. Local: `git push`")
print("   3. Colab: chạy lại Cell 1 (clone fresh)")
print("   4. Chạy lại Cell 3.5 này để verify")

📋 Config hiện tại (các tham số hay đổi giữa các exp):

  use_noise:
    value: false                                          # false → use bounded forget-push
  learning_rate:
    value: 5.0e-4                                         # LoRA tolerates higher lr than full FT
  unlearn_epochs:
    value: 8                                              # LoRA usually converges <10 epochs
  alpha:
    value: 1.0
  beta:
    value: 1.0
  theta:
    value: 0.5
  gamma:
    value: 0.5
  eta_re_anchor:
    value: 0.0                                            # [EXP 002] tắt re-anchor
  kappa_cls_retain:
    value: 1.0                                            # CE on retain (keep classification correct)
  kappa_cls_forget:
    value: 0.5                                            # neg CE on forget (push classification wrong)
  cls_forget_clamp:
    value: 5.0                                            # cap forget CE to avoid divergence
  unfreeze_classifier_heads:
    value: true           

In [ ]:
# ====================================
# CELL 4: Chạy LoKU Unlearning (với auto-tracking)
# ====================================
# ✏️  ĐỔI 2 BIẾN NÀY CHO MỖI EXPERIMENT MỚI:
EXP_NAME   = "exp04_aggressive_neggrad"
HYPOTHESIS = "Exp 03 cho thay UU=15 de bep CLS_frg (chi -0.7 sau weight). Exp 04: giam beta 1.0->0.1 (UU bot manh), tang kappa_cls_forget 0.5->3.0 (gradient ascent gap 6 lan), tang epoch 8->15. Doi: Forget AUC giam ve ~0.65-0.75, MIA giam ve ~0.45-0.55, Test AUC tuc co the giam ve 0.55-0.65 (trade-off chap nhan)."

# (Optional) Đổi sang True nếu muốn xóa checkpoint cũ trước khi train
FRESH_START = True

# ----- Chạy training với auto-tracking -----
fresh_flag = "--fresh" if FRESH_START else ""
!PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py \
    --config config.yaml {fresh_flag} \
    --exp {EXP_NAME} \
    --hypothesis "{HYPOTHESIS}"

print("\n" + "="*60)
print(f"📄 Xem file experiment: experiments/exp_*_{EXP_NAME}.md")
print(f"📊 INDEX (bảng tổng):   experiments/INDEX.md")
print("="*60)
print("👉 Chạy CELL 5 để auto-commit & push lên GitHub")

In [19]:
# ====================================
# CELL 5: Auto-commit & push experiment results lên GitHub
# ====================================
# 3 cách setup credentials (chọn 1, theo độ tiện):
#
# CÁCH A — Lưu vào Drive (KHUYẾN NGHỊ, setup 1 lần dùng mãi):
#   Tạo file /content/drive/MyDrive/Forget-MI-Project/.git-secrets.json
#   với nội dung:
#   {
#     "GITHUB_TOKEN": "ghp_xxxxxxxxxxxx",
#     "GIT_EMAIL": "ban@gmail.com",
#     "GIT_NAME": "Nguyen Hoang Nhu"
#   }
#   Tạo token tại: https://github.com/settings/tokens (scope: repo)
#
# CÁCH B — Colab Secrets (CHỈ web colab.research.google.com):
#   Click 🔑 ở sidebar → Add secret: GITHUB_TOKEN, GIT_EMAIL, GIT_NAME
#
# CÁCH C — Nhập tay mỗi session (lazy, không cần setup):
#   Bỏ qua A và B → cell sẽ tự hỏi token mỗi lần chạy
# ===========================================================
import os, json, getpass
from pathlib import Path

GITHUB_REPO = "nhnhu146/Forget-MI-LoKU"
BRANCH = "master"

def load_secrets():
    # CÁCH A — file trên Drive
    drive_path = Path("/content/drive/MyDrive/Forget-MI-Project/.git-secrets.json")
    if drive_path.exists():
        s = json.loads(drive_path.read_text())
        print(f"🔑 Đã load credentials từ {drive_path}")
        return s.get('GITHUB_TOKEN'), s.get('GIT_EMAIL'), s.get('GIT_NAME')

    # CÁCH B — Colab Secrets (web Colab)
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            print("🔑 Đã load credentials từ Colab Secrets")
            return t, userdata.get('GIT_EMAIL'), userdata.get('GIT_NAME')
    except Exception:
        pass

    # CÁCH B2 — environment variables
    if os.environ.get('GITHUB_TOKEN'):
        print("🔑 Đã load credentials từ env vars")
        return (os.environ['GITHUB_TOKEN'],
                os.environ.get('GIT_EMAIL', ''),
                os.environ.get('GIT_NAME', ''))

    # CÁCH C — nhập tay (fallback)
    print("🔑 Nhập credentials thủ công (sẽ ẩn khi gõ token):")
    print("   (lần sau muốn auto, tạo file Drive theo CÁCH A ở comment trên)")
    t = getpass.getpass("  GitHub token (ghp_...): ").strip()
    e = input("  Git email: ").strip()
    n = input("  Git name:  ").strip()
    return t, e, n


TOKEN, EMAIL, NAME = load_secrets()

if TOKEN and EMAIL and NAME:
    # 1. Configure git identity (chỉ trong repo này, không ảnh hưởng global)
    !git config user.email "{EMAIL}"
    !git config user.name "{NAME}"

    # 2. Inject token vào remote URL (chỉ trong session này)
    !git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git

    # 3. Pull trước để tránh conflict
    !git pull --rebase origin {BRANCH} 2>&1 | tail -5

    # 4. Stage CHỈ file experiment
    !git add experiments/ 2>/dev/null

    # 5. Hiển thị thay đổi
    changes = !git diff --cached --name-only
    if changes and any(c.strip() for c in changes):
        print("\n📦 Files sẽ commit:")
        for f in changes:
            if f.strip():
                print(f"   - {f}")

        commit_msg = f"exp {EXP_NAME}: auto-tracked results"
        !git commit -m "{commit_msg}"
        !git push origin {BRANCH}

        print(f"\n✅ Đã push lên GitHub")
        print(f"🔗 Xem online: https://github.com/{GITHUB_REPO}/tree/{BRANCH}/experiments")
    else:
        print("ℹ️  Không có file experiment mới để commit.")
else:
    print("⚠️  Thiếu credentials — bỏ qua push. Setup theo CÁCH A/B/C ở comment trên.")


📦 Files sẽ commit:
   - experiments/INDEX.md
   - experiments/exp_003_exp03_classifier_unfrozen_neggrad.md
[master fb66d9b] exp exp03_classifier_unfrozen_neggrad: auto-tracked results
 2 files changed, 275 insertions(+)
 create mode 100644 experiments/exp_003_exp03_classifier_unfrozen_neggrad.md
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.46 KiB | 3.46 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/nhnhu146/Forget-MI-LoKU.git
   a907fc8..fb66d9b  master -> master

✅ Đã push lên GitHub
🔗 Xem online: https://github.com/nhnhu146/Forget-MI-LoKU/tree/master/experiments
